<img src="https://www.epfl.ch/about/overview/wp-content/uploads/2020/07/logo-epfl-1024x576.png" width="140px" alt="EPFL_logo">

## Image Processing Laboratory Notebooks
---

This Jupyter notebook is part of a series of computer laboratories which are designed
to teach image-processing programming; they are running on the EPFL's Noto server. They are the practical complement of the theoretical lectures of the EPFL's Master course 
[**MICRO-511 Image Processing I**](https://moodle.epfl.ch/course/view.php?id=522) taught by Prof. M. Unser and Prof. D. Van de Ville.

The project is funded by the Center for Digital Education and the School of Engineering. It is owned by the [Biomedical Imaging Group](https://bigwww.epfl.ch/). 
The distribution or the reproduction of the notebook is strictly prohibited without the written consent of the authors.  © EPFL 2026.

**Authors**: 
    [Pol del Aguila Pla](mailto:pol.delaguilapla@epfl.ch), 
    [Kay Lächler](mailto:kay.lachler@epfl.ch),
    [Alejandro Noguerón Arámburu](mailto:alejandro.nogueronaramburu@epfl.ch),
    [Yan Liu](mailto:yan.liu@epfl.ch), and
    [Daniel Sage](mailto:daniel.sage@epfl.ch).
    
---

To ensure your work is graded correctly by our automated system, **do not create new cells or delete/rearrange/copy existing ones** when you submit. The current cells contain hidden metadata required for the auto-grader to identify your solutions. If you create temporary cells for testing during your work, remember to clean them up before submission.

# Lab 2.2: Filtering Applications
**Released**: Thursday November 5, 2026

**Submission**: Monday November 16, 2026 before 23:59 on [Moodle](https://moodle.epfl.ch/course/view.php?id=522)

**Grade weight**: Lab 2 (16 points), 10% of the overall grade

**Related lectures**: Chapter 3

Double-click on this cell, fill in your name and SCIPER number below, and run the next cell to verify your identity in Noto.

### Student Name:
### SCIPER:

In [ ]:
import getpass

# This line recovers your camipro number to mark the images with your ID
uid = (
    int(getpass.getuser().split("-")[2])
    if len(getpass.getuser().split("-")) > 2
    else ord(getpass.getuser()[0])
)
print(f"SCIPER: {uid}")

## Imports
Run the next cell to import the libraries and load the images that we will use throughout the lab. The images are loaded as `float64`, so that the results of the filters are stored without clipping or rounding.

In [ ]:
# Configure plotting as dynamic
%matplotlib widget

# Import standard required packages for this exercise
import matplotlib.pyplot as plt
import numpy as np
import skimage.io as io
import scipy.ndimage as ndi
import skimage.filters
import ipywidgets as widgets
from interactive_kit import imviewer as viewer

# Load images to be used in this exercise
bikesgray = io.imread("images/bikesgray.tif").astype(np.float64)
camera = io.imread("images/camera-16bits.tif").astype(np.float64)
spots = io.imread("images/spots.tif").astype(np.float64)
hela = io.imread("images/hela-DIC.tif").astype(np.float64)

# Filtering applications (7 points)

After the first part of the lab, we expect you to feel comfortable with the basics of filtering. In this part we will look in detail at the implementation of a Gaussian filter, as well as some of its direct applications. Gaussian filters are known to be near-optimal smoothing filters, and represent perhaps the most used preprocessing step in image processing to improve robustness in a workflow and to denoise images.

### Visualize images
Get familiar with the images that you are going to use by running the next cell.

In [ ]:
# Declare image_list for the viewer
image_list = [bikesgray, camera, spots]

plt.close("all")
imgs_viewer = viewer(image_list, widgets=True, hist=True)

# 1. Gaussian filter

An [isotropic 2D Gaussian](https://en.wikipedia.org/wiki/Multivariate_normal_distribution) is

$$h_\sigma(x,y) = \frac{1}{2\pi\sigma^2}\exp\left(-\frac{x^2 + y^2}{2\sigma^2}\right) \overset{\mathrm{separability}}{=} \frac{1}{\sqrt{2\pi}\sigma}\exp\left(-\frac{x^2}{2\sigma^2}\right) \frac{1}{\sqrt{2\pi}\sigma}\exp\left(-\frac{ y^2}{2\sigma^2}\right) \,,$$

where $\sigma$ is the standard deviation and controls the smoothing strength.

In this section you will implement a 2D Gaussian filter with impulse response $h_{\sigma}[m,n]$, which discretizes $h_\sigma(x,y)$ between $[-\lceil3\sigma\rceil,\lceil 3\sigma\rceil]$ in $x$ and $y$. Here, $\lceil x \rceil$ refers to the smallest integer larger than or equal to a given $x\in\mathbb{R}$. Additionally, choose the size of the filter to be $N = 2\lceil 3\sigma \rceil+1$ (hence, $N$ is always odd), and ensure the impulse response adds up to $1$, using appropriate normalization.

## 1.1. Separable implementation

Since the 2D Gaussian is the product of two 1D Gaussians, it is a separable filter: like in the first part of the lab, we will implement it as two 1D convolutions with the same 1D mask, one along the rows and one along the columns.

### Gaussian mask (1 point)

**For 1 point**, implement the function `gaussian_mask(sigma)` that returns the 1D Gaussian mask of standard deviation `sigma` as a 1D NumPy array of length $N = 2\lceil 3\sigma \rceil+1$, centered at the middle element and normalized so that its elements add up to $1$.

*Hint:* build the array of the positions $x \in \{-\lceil 3\sigma \rceil, \dots, \lceil 3\sigma \rceil\}$ with `np.arange`, evaluate the Gaussian on it with `np.exp`, and divide by the sum. Since the mask is normalized afterwards, the factor $\frac{1}{\sqrt{2\pi}\sigma}$ does not matter.

In [ ]:
# Function that returns the normalized 1D Gaussian mask of standard deviation sigma, as a 1D NumPy array
def gaussian_mask(sigma):
    # Size of the mask (see the formula above)
    N = None
    # Declare the mask
    mask = None
    # YOUR CODE HERE
    return mask

Run the next cell to test your mask on two values of $\sigma$: it checks the length of the mask, its normalization and its symmetry.

In [ ]:
# Visible sanity checks on two values of sigma
for sigma in [0.5, 1]:
    mask = gaussian_mask(sigma)
    N = 2 * int(np.ceil(3 * sigma)) + 1
    assert isinstance(mask, np.ndarray) and mask.ndim == 1, (
        f"gaussian_mask({sigma}) must return a 1D NumPy array, not {type(mask).__name__} of shape {np.shape(mask)}."
    )
    assert len(mask) == N, (
        f"gaussian_mask({sigma}) has {len(mask)} elements, but N = 2 * ceil(3 * sigma) + 1 = {N}. "
        "Did you take the ceiling before multiplying by 2 and adding 1?"
    )
    assert np.isclose(np.sum(mask), 1), f"gaussian_mask({sigma}) adds up to {np.sum(mask):.4f} instead of 1: normalize it."
    assert np.allclose(mask, mask[::-1]) and np.argmax(mask) == N // 2, (
        f"gaussian_mask({sigma}) is not symmetric around its central element: the positions must be centered at zero."
    )
    print(f"sigma = {sigma}: N = {N}, mask = {np.round(mask, 4)}")
print("Your Gaussian mask passed the sanity checks.")

### Filtering (1 point)

**For 1 point**, implement the function `gaussian(img, sigma)` that convolves the image `img` with the 2D Gaussian filter of standard deviation `sigma` using a **separable implementation**: apply the mask returned by `gaussian_mask` along the columns and then along the rows with [`ndi.convolve1d`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.convolve1d.html), exactly like you did for the separable masks of the first part of the lab. Keep the default boundary condition of SciPy (`mode='reflect'`), which repeats the edge pixels.

**Note:** library functions that directly compute a Gaussian filter (such as `ndi.gaussian_filter` or `skimage.filters.gaussian`) are not allowed here: the goal is to build the filter from your own mask. You will compare your implementation with the one of scikit-image in Section 1.3.

In [ ]:
# Function that filters img with a separable Gaussian filter of standard deviation sigma
def gaussian(img, sigma):
    output = np.zeros(img.shape)
    # YOUR CODE HERE
    return output

We have designed a quick test for you to evaluate your method, applying it to a $5 \times 5$ impulse image with $\sigma = 0.5$. Run the following cell and check that your output has all the desired properties of a Gaussian: it should be the $5 \times 5$ outer product of the 1D mask with itself (symmetric, maximal at the center, and adding up to $1$).

In [ ]:
# Define the impulse image
impulse = np.zeros((5, 5))
impulse[2, 2] = 1

# Apply the filter to the impulse
impulse_gaussian = gaussian(impulse, 0.5)
assert impulse_gaussian is not None and np.shape(impulse_gaussian) == (5, 5), (
    "gaussian must return an image of the same shape as its input."
)
print("Your impulse response for sigma = 0.5:")
print(np.round(impulse_gaussian, 4))

# Symmetry: the center is the maximum, and pixels at equivalent positions have the same value
if np.argmax(impulse_gaussian) != 12 or not np.allclose(
    [impulse_gaussian[2, 1], impulse_gaussian[1, 2], impulse_gaussian[3, 2]], impulse_gaussian[2, 3]
):
    print("WARNING!\nThe impulse response is not symmetric around its center: look at it to understand the mistake.")
else:
    print("The symmetry of the Gaussian seems good.")

# Normalization
if not np.isclose(np.sum(impulse_gaussian), 1):
    print(f"WARNING!\nThe impulse response adds up to {np.sum(impulse_gaussian):.4f} instead of 1.")
else:
    print("Well done! The output adds up to 1.")

## 1.2. Multiple choice questions (1 point)

Now that you have tested your Gaussian filter, apply it to the image `bikesgray` with different values of $\sigma$ (you can change them in the next cell, or add more). Look at the evolution of the mean and the standard deviation of the image (you can get them from the statistics box of the `viewer`, or use the functions `np.mean` and `np.std`). Then, answer the two multiple choice questions.

**Note:** large values of $\sigma$ take a while to compute; do not forget to change $\sigma$ back to a normal (small) value after your exploration!

In [ ]:
# Apply your filter with different values of sigma. To try other values, change the variables or declare more.
bikesgray_gaussian1 = gaussian(bikesgray, 1)
bikesgray_gaussian5 = gaussian(bikesgray, 5)

# Declare the parameters of the viewer. If you want to visualize more sigma values, update these lists accordingly
image_list_blur = [bikesgray, bikesgray_gaussian1, bikesgray_gaussian5]
title_list_blur = ["Original", "Sigma: 1", "Sigma: 5"]

# To allow a direct comparison of the images
plt.close("all")
blurred_bikesgray_viewer = viewer(image_list_blur, title=title_list_blur, hist=True)

**For 0.5 points each**, answer the next two questions after modifying the cell above and visualizing the results.

* Q1: How would you expect the Fourier transform of an image to change after applying a Gaussian filter?
    1. It will show lower values for higher frequencies.
    2. It will show higher values for higher frequencies.
    3. It will show lower values for lower frequencies.
    4. It will not change.

* Q2: What will be the output image when $\sigma\rightarrow \infty$? What type of filter would that be?
    1. An image equal to the original. It would be an all-pass filter.
    2. A constant image. It would be a high-pass filter.
    3. A 2D Gaussian. It would be a band-pass filter.
    4. A constant image. It would be a low-pass filter.

Modify the variables `answer_one` and `answer_two` in the next cell to match your choices. The two cells after it are for you to make sure that your answers are in the valid range (they should not print any warning).

In [ ]:
# Modify these variables
answer_one = None
answer_two = None
# YOUR CODE HERE

In [ ]:
# Sanity test
if not answer_one in [1, 2, 3, 4]:
    print("WARNING!\nAnswer one of 1, 2, 3 or 4.")

In [ ]:
# Sanity test
if not answer_two in [1, 2, 3, 4]:
    print("WARNING!\nAnswer one of 1, 2, 3 or 4.")

## 1.3. Gaussian filter in scikit-image (1 point)

There are several implementations of Gaussian filters in Python. In this section, we will use the scikit-image implementation [`skimage.filters.gaussian`](https://scikit-image.org/docs/stable/api/skimage.filters.html#skimage.filters.gaussian). This function is a wrapper around the SciPy function [`scipy.ndimage.gaussian_filter`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.gaussian_filter.html), feel free to check both documentations to inform yourself.

Here is an example of how to use it:

```python
output = skimage.filters.gaussian(input, sigma=10, mode="reflect", truncate=3, preserve_range=True)
```

The above line applies to the `input` image a Gaussian filter with $\sigma=10$, with the edge-repeating boundary condition (`mode="reflect"` in SciPy and scikit-image, see the first part of the lab), truncated to the range $3\sigma$ (i.e. the same size $N = 2\lceil 3\sigma \rceil+1$ as your mask for this value of $\sigma$), and keeping the original range of values of `input` (without this last parameter, integer images are rescaled to $[0, 1]$).

In the next cell, we apply it to the image `bikesgray` and compute your implementation on the same image, also for $\sigma = 10$. Run it to visualize the result of scikit-image.

In [ ]:
# Apply the Gaussian filter of scikit-image
bikesgray_gaussian_skimage = skimage.filters.gaussian(
    bikesgray, sigma=10, mode="reflect", truncate=3, preserve_range=True
)
# Apply your Gaussian filter
bikesgray_gaussian10 = gaussian(bikesgray, 10)

plt.close("all")
gaussian_viewer = viewer(bikesgray_gaussian_skimage, title=["Scikit-image, sigma = 10"])

**For 1 point**, your implementation is compared with the result of scikit-image to make sure that they are equivalent (up to numerical errors on the order of $10^{-12}$). Run the next cell to make the comparison. If the images are not the same, a viewer pops up showing you in red the regions that differ the most: if necessary, use this information to find your mistake.

In [ ]:
# Compare your result with the one of scikit-image
assert (
    bikesgray_gaussian10 is not None
    and np.shape(bikesgray_gaussian10) == bikesgray.shape
), "gaussian must return an image of the same shape as its input."
print(
    f"Maximal absolute difference between your filter and scikit-image: {np.max(np.abs(bikesgray_gaussian10 - bikesgray_gaussian_skimage)):.2e}"
)
plt.close("all")
if not np.allclose(bikesgray_gaussian10, bikesgray_gaussian_skimage):
    print(
        "The results of your Gaussian filter do not match the scikit-image results! Look at the red areas in the viewer to see where you might have gone wrong."
    )
    skimage_gaussian_viewer = viewer(
        [bikesgray_gaussian10, bikesgray_gaussian_skimage],
        title=["Yours", "Scikit-image"],
        widgets=True,
        compare=True,
    )
else:
    print("Seems like your Gaussian filter is correct!")

# 2. Application: segmentation of a DIC image

Now that you have implemented several filters and mastered the concepts behind digital filtering, we are going to show you a real application of Gaussian filtering. In this application, you will work on an image of *DIC microscopy*, and you will use Gaussian filters to estimate the *local standard deviation*.

<div class="alert alert-info">

**Note on DIC:** [Differential interference contrast (DIC) microscopy](https://en.wikipedia.org/wiki/Differential_interference_contrast_microscopy) is a label-free microscopy technique used to visualize transparent samples. It gains information about the optical path length of the sample through interferometry, which allows it to detect the shape of a transparent sample in ways that traditional microscopy cannot.

It produces shadow-like and reflection-like structures. Uniform regions in the sample will produce uniform values, whereas regions with high variations will present variations in the values. However, if we want to detect where our sample is in the image and separate it from the rest of the image, it is impossible to do so using simple thresholding. A classical solution is to threshold the local standard deviation of the input image. With proper values of threshold $t$ and of standard deviation $\sigma$, the sample will be completely extracted from the background.

Take a look at the detailed workflow of *DIC segmentation* in the next figure.

<img src="images/DIC_segmentation_workflow.jpg" width="600px" alt="DIC segmentation workflow">

*Segmentation of a DIC workflow: a Gaussian filter is applied to the original image f, to obtain the local mean m. Then the local s is obtained by subtracting f-m, and applying the square, a Gaussian and the square root operators. The final image is constructed by placing f in the red and green channels, and the thresholded s in the blue channel.*
</div>

## 2.1. Implementation (3 points)

**For 3 points**, implement the three functions `local_mean`, `local_std` and `segment_dic` (each worth $1$ point) that build the algorithm sketched in the previous figure. It displays in the blue channel the thresholded local standard deviation, and in the red and green channels the original image. The two parameters $\sigma$ and $t$ are given as inputs of the workflow.

`local_mean(img, sigma)` takes as parameters:
 * `img`: the original DIC microscopy image, and
 * `sigma`: the standard deviation to use during Gaussian filtering.

It returns the local mean (m in the diagram).

`local_std(img, mean, sigma)` takes as parameters:
 * `img`: the original DIC microscopy image,
 * `mean`: the local mean of the original DIC microscopy image, and
 * `sigma`: the standard deviation to use during Gaussian filtering.

It returns the local $\sigma$ (s in the diagram).

Finally, `segment_dic(img, std, t)` takes as parameters:
 * `img`: the original DIC microscopy image,
 * `std`: the local $\sigma$ of the original DIC microscopy image, and
 * `t`: the threshold, in the range $[0, 255]$.

It returns the segmented RGB image.

*Hint:*
* Keep the final image in the range $[0, 255]$.
* To assign a grayscale $(m\times n)$ image `grayscale_img` to a channel `channel` of an RGB image you can simply do `RGB_img[:, :, channel] = grayscale_img`.
* Use the comments in the function `segment_dic` to guide your solution.
* Make use of the scikit-image Gaussian filter that we presented in Section 1.3. Make sure you use the correct parameters: truncate the filter at $3\sigma$, use the edge-repeating boundary condition `mode="reflect"` and pay attention to the `preserve_range` option.
* For RGB images, matplotlib requires the data to be in the range $[0, 1]$ for images of data type `float`, and between $[0, 255]$ for images of data type `int`. When working with RGB images, make sure to cast your output to one of these data types (e.g. `output = output.astype(int)`) and range. In the following cell we cast the datatype for you, but you have to make sure that the range is appropriate.

In [ ]:
# Complete local_mean to calculate the local mean
def local_mean(img, sigma):
    mean = np.zeros(img.shape)

    # Calculate the local mean
    # YOUR CODE HERE
    return mean

In [ ]:
# Complete local_std to calculate the local standard deviation
def local_std(img, mean, sigma):
    s = np.zeros(img.shape)

    # Estimate the local standard deviation
    # YOUR CODE HERE
    return s

In [ ]:
# Complete segment_dic, that normalizes, binarizes and builds an RGB image
def segment_dic(img, std, t):
    # Declare an image with three channels of the same size as the original
    M, N = img.shape
    output = np.zeros((M, N, 3))

    # Normalize std to the range [0, 255]
    # YOUR CODE HERE

    # Apply thresholding to std: 0 below t, 255 above
    # YOUR CODE HERE

    # Assign the thresholded std to the blue (2) channel, and the original image to the red (0) and green (1) channels
    # YOUR CODE HERE

    # We make sure to have an int type (this operation floors all elements of the output)
    output = output.astype(int)
    return output

Before visualizing the segmentation, we will run a few sanity checks. In the next cell, we will test that:
 * the output is correctly normalized,
 * the channels are indeed the binarized standard deviation (blue channel) and the original image (red and green channels),
 * the standard deviation is correct (using a $3\times 3$ test image).

The three functions are graded separately: the next cell grades `local_mean`, and the two cells of the following sections grade `local_std` (given the correct mean) and `segment_dic` (given the correct standard deviation), $1$ point each.

In [ ]:
# Here we will test the local mean and std on the real image hela
mean = local_mean(hela, 1)
std = local_std(hela, mean, 1)
segmented_hela = segment_dic(hela, std, 10)
assert segmented_hela is not None and np.shape(segmented_hela) == (
    hela.shape[0],
    hela.shape[1],
    3,
), "segment_dic must return an RGB image of shape (M, N, 3)."

# First we test the normalization
if not segmented_hela.min() == 0:
    print(
        f"WARNING!\nYour minimum value is not correct (it should be 0, instead of {segmented_hela.min()})"
    )
elif not segmented_hela.max() == 255:
    print(
        f"WARNING!\nYour maximum value is not correct (it should be 255, instead of {segmented_hela.max()})"
    )
else:
    print("Well done! Your output seems correctly normalized.")

# Now the binarization
if not len(np.unique(segmented_hela[:, :, 2])) == 2:
    print(
        f"WARNING!\nYour blue channel should have 2 values, instead of {len(np.unique(segmented_hela[:, :, 2]))}."
    )

# Now we will check the correctness of the standard deviation with a 3x3 impulse
impulse = np.array([[0, 0, 0], [0, 255, 0], [0, 0, 0]], dtype=np.float64)

# Apply your functions
impulse_mean = local_mean(impulse, 1)
impulse_std = local_std(impulse, impulse_mean, 1)
segmented_impulse = segment_dic(impulse, impulse_std, 10)

# We inspect the blue channel
blue_correct = np.array([[0, 255, 0], [255, 255, 255], [0, 255, 0]])
if not np.count_nonzero(segmented_impulse[:, :, 2] - blue_correct) == 0:
    print("WARNING!\nYour standard deviation does not seem to be correct!")
else:
    print("Good job! Your implementation passed this initial sanity check.")

Now that we have some sanity checks on the segmentation, we will visualize the local mean and $\sigma$. Make sure to understand both plots! Ask yourself, *what does the local mean represent? What does the local $\sigma$ represent? How do you relate your workflow with the [definition](https://en.wikipedia.org/wiki/Standard_deviation#Uncorrected_sample_standard_deviation) of the standard deviation $\sigma$?*

Run the next cell to visualize them (this cell also grades `local_std`, given the correct local mean).

In [ ]:
# Here we visualize the local mean and std
mean = local_mean(hela, 5)
std = local_std(hela, mean, 5)
assert std is not None and np.shape(std) == hela.shape, (
    "local_std must return an image of the same shape as its input."
)
assert not np.any(np.isnan(std)), (
    "Your local standard deviation contains NaN values: you probably took the square root of a negative quantity "
    "(did you square the difference between the image and its local mean before filtering?)."
)
segmented_hela = segment_dic(hela, std, 10)

plt.close("all")
segment_dic_viewer = viewer(
    [hela, mean, std, segmented_hela],
    title=["Original", "Mean", "Standard Dev", "Segmentation"],
    cmap="viridis",
    widgets=True,
    hist=True,
)

## 2.2. Understanding the effect of the parameters

Hopefully, you agree that not a lot of extra information can be extracted from the mean (indeed, it is *just* a Gaussian filtering). But how about the standard deviation? If you implemented the workflow correctly, you should see that it really separates the area of interest from the rest. Let's now look at the role of the two parameters $\sigma$ and $t$.

**Note:** so far, we have not discussed the effects of $\sigma$ and of $t$. We have blindly been using $1$ and $10$ respectively. But what constitutes an appropriate parameter? In the next cell, we prepared an interactive viewer with two sliders in the `Extra widgets` menu:
* the first one controls $\sigma$,
* the second one controls $t$.

Use the button `Segment` to see the effect of the parameters! This cell also grades `segment_dic`, given the correct local standard deviation.

In [ ]:
# Interactive segmentation
sigma_slider = widgets.FloatSlider(
    value=1, min=0.1, max=5, step=0.1, description="sigma"
)
t_slider = widgets.FloatSlider(value=10, min=1, max=255, step=0.1, description="t")
segment_button = widgets.Button(description="Segment")


def segment_callback(img):
    sigma = sigma_slider.value
    t = t_slider.value

    mean = local_mean(hela, sigma)
    std = local_std(hela, mean, sigma)
    segmented_hela = segment_dic(hela, std, t)
    return segmented_hela


plt.close("all")
interactive_dic_viewer = viewer(
    hela,
    title=["Segmentation"],
    new_widgets=[sigma_slider, t_slider, segment_button],
    callbacks=[segment_callback],
    widgets=True,
)
segment_button.click()

🎉 Congratulations on finishing the second part of the Filtering lab!

Make sure to save your notebook (you might want to keep a copy on your personal computer) and upload it to Moodle, **in a zip file with the other notebook of this lab.**

* Keep the name of the notebook as: *2_filtering_applications.ipynb*,
* Name the zip file: *filtering_lab.zip*.